<a href="https://colab.research.google.com/github/mkvkanpur/hpc/blob/main/Struct_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# struct fn

In [20]:
import torch
device = "cuda"

a = torch.tensor([1, 2, 3], dtype=torch.int32)
b = torch.tensor([1, 2, 3], dtype=torch.float32)
amag = a.to(torch.float32).norm()
bmag = b.norm()
res = torch.dot(a.to(torch.float32), b)
print(res.item())
print(a[0].item(), a[1].item())
A = torch.arange(10, dtype=torch.float32, device=device).view(-1, 1)
B = A[1:a[2]]
print(A, '\n')
print(bmag, int(bmag))
force = torch.tensor([1.0, 0.5, -0.2])

# Prepare it for a batch-processing kernel
batch_force = force.unsqueeze(0) # Shape is now [1, 3]
print(batch_force)

l_vect_min = torch.tensor([1,1],dtype=torch.int32).to(device)
print(l_vect_min[0])


14.0
1 2
tensor([[0.],
        [1.],
        [2.],
        [3.],
        [4.],
        [5.],
        [6.],
        [7.],
        [8.],
        [9.]], device='cuda:0') 

tensor(3.7417) 3
tensor([[ 1.0000,  0.5000, -0.2000]])
tensor(1, device='cuda:0', dtype=torch.int32)


In [25]:
import torch
import numpy as np

def compute_relative_average(A, l_vect):
    N = A.shape[0]
    limit = N // 2

    base_quadrant = A[0:limit, 0:limit]

    offset_quadrant = A[l_vect[0] : limit + l_vect[0], l_vect[1] : limit + l_vect[1]]
    diff = offset_quadrant - base_quadrant

    return torch.mean(diff)

@torch.compile
def str_fn(ux_t, uy_t, str_t, l_vect_min, l_vect_max):
  for lx in range(l_vect_min[0], l_vect_max[0]):
    for ly in range(l_vect_min[1], l_vect_max[1]):
      l_vect = torch.tensor([lx, ly]).to(device)
      lmag = l_vect.to(torch.float32).norm()
      index_tensor = lmag.to(torch.int32).unsqueeze(0) # Convert to 1D tensor

      dux = compute_relative_average(ux_t, l_vect)/lmag
      duy = compute_relative_average(uy_t, l_vect)/lmag
      du_vect = torch.stack([dux, duy]).to(device)

      res = torch.dot(l_vect.to(torch.float32), du_vect)
      print("l, res", lx, ly, index_tensor.item(), res.item())

      # Ensure res is also 1D for index_add_
      str_t.index_add_(0, index_tensor, res.unsqueeze(0))
      # res.unsqueeze(0) makes it 1D
  return # Move return outside the loop

# --- Test Case ---
N = 10
device = "cuda" if torch.cuda.is_available() else "cpu"

ux = np.arange(N, dtype=np.float32).reshape(-1, 1) * np.ones((1, N), dtype=np.float32)
uy = np.arange(N, dtype=np.float32).reshape(1, -1) * np.ones((N, 1), dtype=np.float32)
str = np.zeros(N, dtype=np.float32)

ux_t = torch.from_numpy(ux).to(device)
uy_t = torch.from_numpy(uy).to(device)
str_t = torch.from_numpy(str).to(device)

l_vect_min = np.array([1,1])
l_vect_max = np.array([4,4])

dux = compute_relative_average(ux_t,l_vect)
duy = compute_relative_average(uy_t,l_vect)

du_vect = torch.stack([dux, duy]).to(device)
res = torch.dot(l_vect.to(torch.float32), du_vect)

str_fn(ux_t, uy_t, str_t, l_vect_min, l_vect_max)
print(f"str_t: {str_t}")

l, res 1 1 1 1.4142135381698608
l, res 1 2 2 2.2360680103302
l, res 1 3 3 3.1622774600982666
l, res 2 1 2 2.2360680103302
l, res 2 2 2 2.8284270763397217
l, res 2 3 3 3.605551242828369
l, res 3 1 3 3.1622774600982666
l, res 3 2 3 3.605551242828369
l, res 3 3 4 4.242640972137451
str_t: tensor([ 0.0000,  1.4142,  7.3006, 13.5357,  4.2426,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000], device='cuda:0')
